# 01 — Train the 1D CNN

**Purpose:** train a CREB1 binding classifier on the dataset produced by `00_data_sanity.ipynb`.

This notebook is a **thin consumer** of `src/`. All training logic lives in `src/train.py`, so the same code can be run from the terminal (`python -m src.train`) or in this notebook. If you find yourself writing a training loop here, put it in `src/train.py` instead.

**Model:** `Creb1CNN` from `src/model.py` — a 3-block 1D CNN with adaptive average pooling, producing a single logit per sequence.

**Training config:**
- Loss: `BCEWithLogitsLoss` (sigmoid + BCE, numerically stable)
- Optimizer: Adam, lr = 1e-3
- Batch size: 64
- Epochs: 20
- Split: test chromosomes `1, 2`; validation is a random 10% of the training rows (this is for monitoring only — the leak-free split is train↔test).

**Outputs:** `checkpoints/cnn.pt` (best-val-loss checkpoint) and an in-memory loss history.

In [ ]:
import sys, os
from pathlib import Path

p = Path.cwd()
while p != p.parent and not (p / "src").exists():
    p = p.parent
os.chdir(p)
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print("repo root:", p)

from src.notebook_setup import set_repo_root
set_repo_root()

In [ ]:
from src.notebook_setup import set_repo_root
set_repo_root()

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data import DataConfig, build_dataset, split_by_chromosome
from src.model import Creb1CNN
from src.train import make_loaders, train, save_checkpoint

## 1. Configuration

All hyperparameters are plain variables here, edit and re-run the training cell without touching `src/`.

**Note on `test_chroms`:** chromosome-level holdout is the standard defense against near-duplicate leakage in genomics models. `("1", "2")` follows the convention used by Basenji, Enformer, and most TF-binding benchmarks.

**Note on the validation split:** validation is a *random row split* within the training chromosomes, not a chromosome holdout. This is intentional — val loss is for monitoring overfitting during training, not for reporting final numbers. The leak-free number is the test AUROC computed in `02_eval.ipynb`.

In [ ]:
cfg = DataConfig(window=500, neg_per_pos=1, seed=0, test_chroms=("1", "2"))
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = "checkpoints/cnn.pt"

print(f"device: {DEVICE}")
print(f"config: {cfg}")

## 2. Load the cached dataset

`build_dataset` loads from `data/processed/*.npz` if the cache exists. This should take under a second — if it takes minutes, the cache is missing and the pipeline is rebuilding from the FASTA.

The three-way split below is:
- **train / val** — both from training chromosomes (`3`–`22`, `X`, `Y`), val is a random 10% of rows
- **test** — held out chromosomes `1` and `2`, used only in `02_eval.ipynb`

**What to check:** shapes are `(N, 4, 500)` and positive rates are ≈ 0.5 across all three splits.

In [ ]:
data = build_dataset(cfg)
split = split_by_chromosome(data, test_chroms=cfg.test_chroms)

rng = np.random.default_rng(cfg.seed)
val_mask = rng.random(len(split["y_train"])) < 0.1

X_tr = split["X_train"][~val_mask]; y_tr = split["y_train"][~val_mask]
X_va = split["X_train"][val_mask];  y_va = split["y_train"][val_mask]
X_te = split["X_test"]; y_te = split["y_test"]

print(f"train: {X_tr.shape}  val: {X_va.shape}  test: {X_te.shape}")
print(f"pos rate — train {y_tr.mean():.3f}  val {y_va.mean():.3f}  test {y_te.mean():.3f}")

## 3. Build the model

`Creb1CNN` is a 3-block 1D CNN:
- **Block 1:** 2× `Conv1d(4→64, k=11, same)` + `MaxPool(4)`
- **Block 2:** 2× `Conv1d(64→128, k=11, same)` + `MaxPool(4)`
- **Block 3:** 2× `Conv1d(128→256, k=11, same)`
- **Head:** `AdaptiveAvgPool1d(1)` → `Dropout(0.3)` → `Linear(256→1)`

The `AdaptiveAvgPool1d` makes the head independent of input length — you can change `cfg.window` without touching the model.

**Shape sanity check:** the output must be `(batch,)`, *not* `(batch, 1)`. A `(batch, 1)` output broadcast against `(batch,)` labels produces a `(batch, batch)` loss that looks plausible but trains nothing. This is the most common silent bug in PyTorch sequence models — the check below catches it.

In [ ]:
train_loader, val_loader = make_loaders(
    X_tr, y_tr, X_va, y_va,
    batch_size=BATCH_SIZE, seed=cfg.seed,
)

model = Creb1CNN()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    print(f"output shape: {model(torch.zeros(2, 4, cfg.window)).shape}")   # must be (2,)

## 4. Train

Everything below runs inside `src.train.train()`. This cell is three lines for a reason: the training loop, optimizer setup, loss function, and checkpointing are all in `src/`, so the same behavior is reproducible from the terminal.

**What to watch:**
- Both losses should decrease. If train loss is flat at `ln(2) ≈ 0.693`, the model isn't learning, go back and re-check the output shape in cell 8.
- If val loss starts rising while train keeps falling, you're overfitting. The checkpoint saved is the **best-val-loss** epoch, not the last epoch, so the saved model is the right one to evaluate.

In [ ]:
history = train(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, device=DEVICE, ckpt_path=CKPT,
)
print(f"checkpoint saved to {CKPT}")

## 5. Training curves

Visualizes train vs validation loss over epochs.

**What to look for:**
- The gap between train and val loss should stay small. A widening gap is overfitting.
- Both curves should plateau by the end of training — if they're still dropping steeply, you can train longer.
- A sudden spike in either curve usually means a learning rate that's too high, or a single bad batch (shuffle seed issue).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["train_loss"], label="train", marker="o", markersize=3)
ax.plot(history["val_loss"], label="val", marker="o", markersize=3)
ax.set_xlabel("epoch")
ax.set_ylabel("BCE loss")
ax.set_title("Training curve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Done

The checkpoint at `checkpoints/cnn.pt` is the best-val-loss model from this run. It's what `02_eval.ipynb` and `python -m src.evaluate` load.

**Do not read test metrics in this notebook.** Test evaluation happens in `02_eval.ipynb`, separating training from evaluation prevents the subtle failure mode of "tuning against test set."